In [ ]:
import sys, os
# Adjust depth based on notebook location relative to project root
_src_path = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']
well_meta = config.get('well_metadata', {})

print(f'Active experiment: {config.get("experiment_name", config.get("experiment_key", "?"))}')
print(f'Base dir: {base_dir}')

In [ ]:
# Batch Feature-Pair PowerPoint Builder (no UMAP/PHATE)
# Processes every .h5ad file in a folder and builds PPTX reports.
#
# What you get per file:
# - Rank all feature–feature pairs (Spearman + BH-FDR), ordered by significance
# - Laplacian score per feature (lower = more informative)
# - One slide per top pair with:
#     * Scatter (A vs B) colored by best KMeans (K via silhouette)
#     * Violin(A) by obs[OBS_SPLIT_KEY]
#     * Violin(B) by obs[OBS_SPLIT_KEY]
# - Titles include stats & file name for Ctrl+F
#
# Deps: numpy pandas scipy scikit-learn matplotlib python-pptx anndata

# ======== CONFIG (EDIT THESE) ========
INPUT_DIR = r"D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025"
OUTPUT_MODE = "separate"  # "separate" = one PPT per .h5ad; "single" = one combined PPT
COMBINED_PPTX = "combined_feature_pairs_no_embed.pptx"  # used if OUTPUT_MODE == "single"

OBS_SPLIT_KEY = "treatment"     # e.g., "treatment" or "sample_ID"
TOP_FEATURES_BY_LAPLACIAN = 34  # None to use all features in each file
TOP_PAIRS = None                # None for all pairs, or set an int (e.g., 100)
K_RANGE = range(2, 7)           # KMeans K search range
RANDOM_STATE = 42               # for KMeans etc.
FIG_DPI = 150                   # lower -> faster/smaller files (e.g., 120)
# ====================================

import os, gc, glob, tempfile
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # comment this out if you want inline plots in Jupyter
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_AUTO_SHAPE_TYPE

import anndata as ad

def stamp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def clean_dense_array(X: np.ndarray) -> np.ndarray:
    """Ensure finite values, impute NaNs with column medians, clip 99.9% for robustness."""
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = np.nan
    col_med = np.nanmedian(X, axis=0)
    col_med[~np.isfinite(col_med)] = 0.0
    r, c = np.where(~np.isfinite(X))
    if r.size:
        X[r, c] = col_med[c]
    hi = np.nanquantile(X, 0.999, axis=0)
    X = np.minimum(X, hi)
    return X

def laplacian_scores(X: np.ndarray, k: int = 30) -> np.ndarray:
    """Simple Laplacian score per feature; lower is better."""
    if X.ndim != 2: X = np.atleast_2d(X)
    k_safe = min(k, max(1, X.shape[0]-1))
    W = kneighbors_graph(X, n_neighbors=k_safe, mode="connectivity", include_self=False).tocsr()
    d = np.asarray(W.sum(axis=1)).ravel()
    scores = np.zeros(X.shape[1], dtype=float)
    for j in range(X.shape[1]):
        f = X[:, j]
        f = f - np.average(f, weights=d+1e-12)
        fDf = (d * (f**2)).sum()
        fWf = f @ (W @ f)
        num = fDf - fWf
        den = (d * (f**2)).sum() + 1e-12
        scores[j] = num / den
    return scores

def benjamini_hochberg(pvals: np.ndarray) -> np.ndarray:
    """Benjamini–Hochberg FDR correction."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    rank = np.empty_like(order)
    rank[order] = np.arange(1, n+1)
    q = p * n / rank
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    q_vals = np.empty_like(q_sorted)
    q_vals[order] = q_sorted
    return np.clip(q_vals, 0, 1)

def best_kmeans_on_pair(x: np.ndarray, y: np.ndarray, k_range=K_RANGE, random_state=RANDOM_STATE):
    """KMeans on 2D pair; return labels, best_k, best_silhouette."""
    XY = np.vstack([x, y]).T
    XYz = StandardScaler().fit_transform(XY)
    best_k, best_sil, best_labels = None, -np.inf, None
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        labels = km.fit_predict(XYz)
        if len(np.unique(labels)) < 2:
            continue
        sil = silhouette_score(XYz, labels)
        if sil > best_sil:
            best_k, best_sil, best_labels = k, sil, labels
    if best_labels is None:
        km = KMeans(n_clusters=2, n_init=10, random_state=random_state)
        best_labels = km.fit_predict(XYz)
        best_k, best_sil = 2, float("nan")
    return best_labels, best_k, best_sil

def ensure_dir(path: str):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def build_deck_for_file(h5ad_path: str, prs: Presentation = None) -> Presentation:
    """Build slides for one .h5ad. If prs provided, append to it; else create new deck."""
    fname = os.path.basename(h5ad_path)
    stamp(f"Loading AnnData: {h5ad_path}")
    adata = ad.read_h5ad(h5ad_path)
    X = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X.copy()
    X = clean_dense_array(X)
    feature_names = np.array(adata.var_names, dtype=str)

    # Laplacian selection or context
    if TOP_FEATURES_BY_LAPLACIAN is not None and TOP_FEATURES_BY_LAPLACIAN < X.shape[1]:
        stamp("Selecting top features by Laplacian score...")
        l_scores = laplacian_scores(StandardScaler().fit_transform(X), k=30)
        order = np.argsort(l_scores)  # lower better
        keep_idx = order[:TOP_FEATURES_BY_LAPLACIAN]
        X = X[:, keep_idx]
        feature_names = feature_names[keep_idx]
        lap_map = {fn: l_scores[i] for fn, i in zip(feature_names, keep_idx)}
    else:
        stamp("Computing Laplacian scores for context...")
        l_scores_all = laplacian_scores(StandardScaler().fit_transform(X), k=30)
        lap_map = {fn: l_scores_all[i] for i, fn in enumerate(feature_names)}

    # Pairwise stats
    stamp("Computing pairwise Spearman correlations...")
    m = X.shape[1]
    pairs_idx, rhos, pvals = [], [], []
    for i in range(m):
        for j in range(i+1, m):
            rho, p = spearmanr(X[:, i], X[:, j])
            pairs_idx.append((i, j))
            rhos.append(rho)
            pvals.append(p)
    rhos = np.array(rhos, dtype=float)
    pvals = np.array(pvals, dtype=float)
    fdr = benjamini_hochberg(pvals)

    rows = []
    for (i, j), rho, p, q in zip(pairs_idx, rhos, pvals, fdr):
        fa, fb = feature_names[i], feature_names[j]
        rows.append({
            "feat_a": fa, "feat_b": fb,
            "rho": rho, "pval": p, "fdr": q,
            "lap_a": lap_map.get(fa, np.nan),
            "lap_b": lap_map.get(fb, np.nan),
            "abs_rho": abs(rho)
        })
    rank_df = pd.DataFrame(rows).sort_values(["fdr", "abs_rho"], ascending=[True, False]).reset_index(drop=True)
    stamp("Top 5 pairs:")
    stamp(rank_df.head().to_string(index=False))

    # Figure cache dir
    tmp_dir = tempfile.mkdtemp(prefix="pairviz_")
    fig_dir = os.path.join(tmp_dir, "figs")
    ensure_dir(fig_dir)

    # Deck setup
    new_deck = False
    if prs is None:
        prs = Presentation()
        prs.core_properties.title = "Feature Pair Relationships (No Embedding)"
        new_deck = True

    # File heading slide (helps Ctrl+F and keeps files separated in combined mode)
    title_slide = prs.slides.add_slide(prs.slide_layouts[0])
    title_slide.shapes.title.text = f"{fname} — Feature Pair Relationships (FDR then |ρ|)"
    subtitle = title_slide.placeholders[1].text_frame
    subtitle.text = f"obs['{OBS_SPLIT_KEY}'] for violins | No UMAP/PHATE"
    p = subtitle.add_paragraph(); p.text = f"Features used: {len(feature_names)}"

    # TOC for this file
    toc_slide = prs.slides.add_slide(prs.slide_layouts[1])
    toc_slide.shapes.title.text = f"Top Relationships — {fname}"
    tf = toc_slide.shapes.placeholders[1].text_frame; tf.clear()

    sel_df = rank_df if TOP_PAIRS is None else rank_df.head(TOP_PAIRS)
    for _, r in sel_df.iterrows():
        para = tf.add_paragraph()
        para.text = f"{r['feat_a']} vs {r['feat_b']} | ρ={r['rho']:.3f}, p={r['pval']:.2e}, FDR={r['fdr']:.2e}"
        para.level = 1

    # Violin split key
    obs = adata.obs
    if OBS_SPLIT_KEY not in obs.columns:
        stamp(f"WARNING: obs['{OBS_SPLIT_KEY}'] not found; violins will use a single group.")
    split_vals = obs[OBS_SPLIT_KEY] if OBS_SPLIT_KEY in obs.columns else pd.Series(["all"] * adata.n_obs)

    # Slides
    stamp(f"Building slides for {len(sel_df)} pairs…")
    for idx, row in sel_df.iterrows():
        if idx % 10 == 0:
            stamp(f"  … {idx}/{len(sel_df)}")
        fa, fb = row["feat_a"], row["feat_b"]
        ia = int(np.where(feature_names == fa)[0][0])
        ib = int(np.where(feature_names == fb)[0][0])
        xa = X[:, ia]; xb = X[:, ib]
        print("1")

        # Best KMeans on this 2D pair
        labels, best_k, best_sil = best_kmeans_on_pair(xa, xb, k_range=K_RANGE, random_state=RANDOM_STATE)
        print("2")

        # Scatter A vs B colored by KMeans
        fig1 = plt.figure(figsize=(6, 5), dpi=FIG_DPI)
        ax = plt.gca()
        sc1 = ax.scatter(xa, xb, c=labels, s=6)
        ax.set_xlabel(fa); ax.set_ylabel(fb)
        ax.set_title(f"Scatter: {fa} vs {fb} (K={best_k}, silhouette={best_sil:.3f})")
        scatter_path = os.path.join(fig_dir, f"scatter_{fa}_vs_{fb}.png")
        plt.tight_layout(); fig1.savefig(scatter_path); plt.close(fig1)
        print("3")

        # Violin A
        fig2 = plt.figure(figsize=(6, 5), dpi=FIG_DPI)
        ax2 = plt.gca()
        groups_a, labels_a = [], []
        cats = list(pd.Categorical(split_vals).categories) if hasattr(split_vals, "dtype") else sorted(set(split_vals))
        for g in cats:
            mask = (split_vals.astype(str).values == str(g))
            if np.sum(mask) > 0:
                groups_a.append(xa[mask]); labels_a.append(str(g))
        if not groups_a:
            groups_a, labels_a = [xa], ["all"]
        vp = ax2.violinplot(groups_a, showmeans=False, showmedians=True, showextrema=False)
        ax2.set_xticks(np.arange(1, len(labels_a)+1)); ax2.set_xticklabels(labels_a, rotation=20, ha="right")
        ax2.set_ylabel(fa); ax2.set_title(f"Violin: {fa} by {OBS_SPLIT_KEY}")
        lo, hi = np.nanpercentile(np.concatenate(groups_a), [1, 99]); ax2.set_ylim(lo, hi)
        violin_a_path = os.path.join(fig_dir, f"violin_{fa}.png")
        plt.tight_layout(); fig2.savefig(violin_a_path); plt.close(fig2)

        # Violin B
        fig3 = plt.figure(figsize=(6, 5), dpi=FIG_DPI)
        ax3 = plt.gca()
        groups_b, labels_b = [], []
        for g in cats:
            mask = (split_vals.astype(str).values == str(g))
            if np.sum(mask) > 0:
                groups_b.append(xb[mask]); labels_b.append(str(g))
        if not groups_b:
            groups_b, labels_b = [xb], ["all"]
        vp = ax3.violinplot(groups_b, showmeans=False, showmedians=True, showextrema=False)
        ax3.set_xticks(np.arange(1, len(labels_b)+1)); ax3.set_xticklabels(labels_b, rotation=20, ha="right")
        ax3.set_ylabel(fb); ax3.set_title(f"Violin: {fb} by {OBS_SPLIT_KEY}")
        lo, hi = np.nanpercentile(np.concatenate(groups_b), [1, 99]); ax3.set_ylim(lo, hi)
        violin_b_path = os.path.join(fig_dir, f"violin_{fb}.png")
        plt.tight_layout(); fig3.savefig(violin_b_path); plt.close(fig3)

        # Add slide
        layout = prs.slide_layouts[5]  # Title Only
        slide = prs.slides.add_slide(layout)
        title_shape = slide.shapes.title
        title_text = (f"{fname} | {fa} vs {fb} | ρ={row['rho']:.3f}, p={row['pval']:.2e}, FDR={row['fdr']:.2e} | "
                      f"Lap({fa})={row['lap_a']:.3g}, Lap({fb})={row['lap_b']:.3g}")
        title_shape.text = title_text  # plain text -> searchable

        # Place images in 1x3 grid
        left = Inches(0.5); top = Inches(1.2)
        img_w = Inches(3.8); img_h = Inches(3.4)
        gap = Inches(0.3)

        slide.shapes.add_picture(scatter_path, left, top, width=img_w, height=img_h)
        slide.shapes.add_picture(violin_a_path, left + img_w + gap, top, width=img_w, height=img_h)
        slide.shapes.add_picture(violin_b_path, left + 2*(img_w + gap), top, width=img_w, height=img_h)

        # Legend box
        box_left = left
        box_top = top + img_h + Inches(0.1)
        box = slide.shapes.add_shape(MSO_AUTO_SHAPE_TYPE.RECTANGLE, box_left, box_top, Inches(12), Inches(0.6))
        tfb = box.text_frame
        tfb.text = f"KMeans on ({fa},{fb}): best K={best_k}, silhouette={best_sil:.3f}"
        tfb.paragraphs[0].font.size = Pt(12)

        gc.collect()

    return prs

# === Run over folder ===
stamp(f"Scanning for .h5ad in: {INPUT_DIR}")
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.h5ad")))
if not files:
    raise FileNotFoundError("No .h5ad files found in the folder.")

stamp(f"Found {len(files)} file(s). Output mode: {OUTPUT_MODE}")

if OUTPUT_MODE == "separate":
    for fp in files:
        prs = build_deck_for_file(fp)
        out_name = os.path.splitext(os.path.basename(fp))[0] + "_feature_pairs_no_embed.pptx"
        out_path = os.path.join(INPUT_DIR, out_name)
        prs.save(out_path)
        stamp(f"Saved PowerPoint: {out_path}")
else:
    combined = Presentation()
    combined.core_properties.title = "Combined Feature Pair Relationships (No Embedding)"
    for fp in files:
        combined = build_deck_for_file(fp, prs=combined)
    out_path = os.path.join(INPUT_DIR, COMBINED_PPTX)
    combined.save(out_path)
    stamp(f"Saved combined PowerPoint: {out_path}")

stamp("Done.")


[13:49:40] Scanning for .h5ad in: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025
[13:49:40] Found 2 file(s). Output mode: separate
[13:49:40] Loading AnnData: D:\Sam\4i_Liposarcoma_Data_YR2025\LPS_246_Abema_July_2025\standard_adata.h5ad
[13:49:41] Selecting top features by Laplacian score...
[13:58:20] Computing pairwise Spearman correlations...
[13:58:43] Top 5 pairs:
[13:58:43]                 feat_a                feat_b      rho  pval  fdr    lap_a    lap_b  abs_rho
 R0_pRb_nuc_cyto_ratio       R0_pRb_nuc_mean 0.987103   0.0  0.0 0.097207 0.098197 0.987103
       R0_DNA_nuc_mean R0_DNA_nuc_cyto_ratio 0.969378   0.0  0.0 0.127653 0.134566 0.969378
       R3_DNA_nuc_mean       R2_DNA_nuc_mean 0.949542   0.0  0.0 0.127186 0.135265 0.949542
R5_Cdt1_nuc_cyto_ratio      R5_Cdt1_nuc_mean 0.943077   0.0  0.0 0.103450 0.109804 0.943077
       R2_DNA_nuc_mean R2_DNA_nuc_cyto_ratio 0.942315   0.0  0.0 0.135265 0.138064 0.942315
[13:58:43] Building slides for 561 pairs…
[13:58:43]  